# Sionna OFDM (orthogonal frequency-division multiplexing) grid channel estimation on Colab

Runs the Sionna resource-grid baselines and the small CNN (convolutional neural network) comparison on a
Colab GPU (graphics processing unit). Set the runtime to GPU (`Runtime -> Change runtime type -> T4 GPU`)
before running.

## 1. GPU (graphics processing unit) check

In [ ]:
import subprocess

try:
    import torch
    print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
except ImportError:
    print("torch is not installed yet; continue to the install cell.")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true

## 2. Clone the repository

In [ ]:
REPO = "/content/sionna-neural-ofdm-channel-estimation"
!rm -rf "$REPO"
!git clone --branch main --single-branch https://github.com/s6ii5vii/sionna-neural-ofdm-channel-estimation.git "$REPO"
%cd "$REPO"
!git rev-parse HEAD

## 3. Install the ml stack

If the import cell below fails, run `Runtime -> Restart session` once, then
re-run from this install cell onward.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import signal
import subprocess
import sys

REPO = Path("/content/sionna-neural-ofdm-channel-estimation")
os.chdir(REPO)

def _version_tuple(value):
    return tuple(int(part) for part in value.split("+")[0].split(".")[:3])

def _dependencies_ready():
    try:
        numpy_version = metadata.version("numpy")
        numba_version = metadata.version("numba")
        torch_version = metadata.version("torch")
        sionna_version = metadata.version("sionna")
        metadata.version("pytest")
    except metadata.PackageNotFoundError:
        return False
    return (
        numpy_version == "2.2.6"
        and (0, 60, 0) <= _version_tuple(numba_version) < (0, 62, 0)
        and _version_tuple(torch_version) >= (2, 9, 0)
        and _version_tuple(sionna_version) >= (2, 0, 0)
    )

if not _dependencies_ready():
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--force-reinstall",
        "numpy==2.2.6",
        "numba>=0.60,<0.62",
    ])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,test]"])
    print("Dependencies installed. Restarting the Colab runtime once; rerun this cell, then continue.")
    try:
        from google.colab import runtime
        runtime.restart_runtime()
        raise SystemExit("runtime restart requested; rerun this cell after reconnecting")
    except Exception:
        os.kill(os.getpid(), signal.SIGKILL)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."])
import numba
import numpy as np
print("Dependencies ready:", "numpy", np.__version__, "| numba", numba.__version__)
print("Repository revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import sionna, torch
print("sionna", sionna.__version__, "| torch", torch.__version__)
from sionna.phy import config
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LSChannelEstimator
from sionna.phy.channel import ApplyOFDMChannel, GenerateOFDMChannel
from sionna.phy.channel.tr38901 import TDL
from sionna.phy.mapping import QAMSource

## 4. Tests

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
!python -m pytest -q

## 5. Least-squares grid baseline (LS-nn (least-squares with nearest-neighbor interpolation) vs LS-lin (least-squares with linear interpolation))

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
!python experiments/grid-tdl-v1/run-experiment.py

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv("results/tables/grid-tdl-v1.csv"))
display(Image("results/figures/grid-tdl-v1-nmse.png"))

## 6. Train the CNN (convolutional neural network) and compare against LS (least-squares)

Training auto-generates the grid dataset if it does not already exist.

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
!python -m channel_estimation.train experiments/grid-neural-comparison-v1/config.yaml

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
import json
print(json.dumps(json.load(open("results/checkpoints/grid-neural-comparison-v1.report.json")), indent=2))

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
!python experiments/grid-neural-comparison-v1/run-experiment.py

In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
display(pd.read_csv("results/tables/grid-neural-comparison-v1.csv"))
display(Image("results/figures/grid-neural-comparison-v1-nmse.png"))

## 7. Optional validation sweep

Run this after the quick comparison if you want stronger evidence across multiple random seeds and TDL (tapped delay line) channel models. The sweep can take noticeably longer than the single-run demo.


In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
!python experiments/grid-neural-comparison-v1/run-sweep.py


In [ ]:
%cd "/content/sionna-neural-ofdm-channel-estimation"
display(pd.read_csv("results/tables/grid-neural-comparison-v1-sweep-summary-v1.csv"))
display(pd.read_csv("results/tables/grid-neural-comparison-v1-sweep-margins-v1.csv"))
display(Image("results/figures/grid-neural-comparison-v1-sweep-v1-nmse.png"))
